The paper's original metric compares SAE features using cosine similarity between encoder or decoder weight vectors, followed by Hungarian matching.

This notebook tries an alternative metric: **co-occurrence matrix + Jaccard similarity matrix**:

- Two features are similar if they fire on the same token positions.
- A co-occurrence matrix records how often pairs occur together. Jaccard normalizes those raw co-occurrence counts into a similarity score.

**Pythia-160M**, the base language model.
   - At the layer-6 MLP hookpoint, each token has a 768-dimensional activation vector.

**SAE 1**
   - It maps a 768-dimensional Pythia activation into 32,768 possible SAE latents/features.
   - Because this is a TopK SAE with k=32, only 32 latents fire per token.

**SAE 2**, same setup as SAE 1 but different initialization seed.


### Work flow:

text

  &darr;

Pythia tokenizer

  &darr;

Pythia-160M forward pass

  &darr;

capture layer 6 MLP activations, [sequence_length, 768]

  &darr;

remove padding positions and flatten, [num_tokens, 768]

  &darr;

feed the exact same activations to:

SAE1.encode()

SAE2.encode()

  &darr;

`top_indices` and `top_acts`, [num_tokens, 32] for each SAE

- `top_indices`: which 32 SAE features fired for each token.
- `top_acts`: how strongly those 32 features fired.

&darr;

convert `top_indices` into binary firing matrices A, B: [num_tokens, num_features (32768)]

(32768 = number of feature columns available

32 = number of those columns set to 1 (i.e. fired) in each row)



raw cross-SAE co-occurrence matrix

$C = A^T B$

$C[i, j]$ = number of tokens where SAE 1 feature i and SAE 2 feature j both fired

  &darr;


compute firing counts for each SAE

$n_A[i], n_B[j]$

  &darr;

Jaccard similarity matrix J 

$J[i, j] =
C[i, j] /
(n_A[i] + n_B[j] - C[i, j])$

  &darr;

Hungarian matching


In [1]:
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment
from sparsify import Sae

print("torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("number of visible GPUs:", torch.cuda.device_count())

/mnt/ssd-2/soar-seed/tianai/.venv_seed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.13.0+cu126
CUDA runtime: 12.6
CUDA available: True
number of visible GPUs: 8


In [2]:
GPU_ID = 0
DEVICE = torch.device(f"cuda:{GPU_ID}") 
MODEL_DEVICE = torch.device("cpu")

### 1. Load SAEs

In [3]:
SAE1_PATH = "/mnt/ssd-2/soar-seed/tianai/checkpoints/sae-pythia-160m-32k/layers.6.mlp"
SAE2_PATH = "/mnt/ssd-2/soar-seed/tianai/checkpoints/sae_overlap/k32-sae-mlp-32k-seed2/layers.6.mlp"

sae1 = Sae.load_from_disk(SAE1_PATH, device=DEVICE)
sae2 = Sae.load_from_disk(SAE2_PATH, device=DEVICE)

print("SAE 1 class:", type(sae1))
print("SAE 2 class:", type(sae2))
print("SAE 1 decoder dictionary:", sae1.W_dec.shape, sae1.W_dec.device, sae1.W_dec.dtype)
print("SAE 2 decoder dictionary:", sae2.W_dec.shape, sae2.W_dec.device, sae2.W_dec.dtype)
print("SAE 1 encoder weights:", sae1.encoder.weight.shape)
print("SAE 2 encoder weights:", sae2.encoder.weight.shape)
print("SAE 1 config:", sae1.cfg)
print("SAE 2 config:", sae2.cfg)

Dropping extra args {'signed': False}


SAE 1 class: <class 'sparsify.sparse_coder.SparseCoder'>
SAE 2 class: <class 'sparsify.sparse_coder.SparseCoder'>
SAE 1 decoder dictionary: torch.Size([32768, 768]) cuda:0 torch.float32
SAE 2 decoder dictionary: torch.Size([32768, 768]) cuda:0 torch.float32
SAE 1 encoder weights: torch.Size([32768, 768])
SAE 2 encoder weights: torch.Size([32768, 768])
SAE 1 config: SparseCoderConfig(activation='topk', expansion_factor=32, normalize_decoder=True, num_latents=32768, k=32, multi_topk=False, skip_connection=False, transcode=False)
SAE 2 config: SparseCoderConfig(activation='topk', expansion_factor=32, normalize_decoder=True, num_latents=32768, k=32, multi_topk=False, skip_connection=False, transcode=False)


### 2. Load Pythia-160M and capture the layer-6 MLP output

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "EleutherAI/pythia-160m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, attn_implementation="eager").to(MODEL_DEVICE)
model.eval()

print("Pythia hidden size:", model.config.hidden_size)
print("Pythia device:", next(model.parameters()).device)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9463.05it/s]


Pythia hidden size: 768
Pythia device: cpu


In [5]:
for name, module in model.named_modules():
    if "mlp" in name and "6" in name:
        print(name, type(module).__name__)

gpt_neox.layers.6.post_mlp_dropout Dropout
gpt_neox.layers.6.mlp GPTNeoXMLP
gpt_neox.layers.6.mlp.dense_h_to_4h Linear
gpt_neox.layers.6.mlp.dense_4h_to_h Linear
gpt_neox.layers.6.mlp.act GELUActivation


In [6]:
captured_activations = {}

def save_layer_6_mlp_output(module, module_inputs, module_output):
    captured_activations["layers.6.mlp"] = module_output.detach()

hook_handle = model.gpt_neox.layers[6].mlp.register_forward_hook(
    save_layer_6_mlp_output
)

### 3. Small batch run

In [7]:
texts = [
    "The cat sat on the warm windowsill and watched the birds outside.",
    "Dogs are loyal companions who love going for walks in the park.",
    "The president signed the new trade agreement with several countries.",
    "Photosynthesis converts sunlight into chemical energy in plant cells.",
    "She finished her PhD thesis on quantum entanglement last Tuesday.",
    "The stock market dropped sharply after the earnings report came out.",
    "Mix the flour and eggs together, then add sugar and vanilla extract.",
    "The astronauts repaired the solar panels during their spacewalk.",
    "Heavy rain caused flooding in several neighborhoods near the river.",
    "The neural network learned to classify images with high accuracy.",
    "Mozart composed his first symphony when he was only eight years old.",
    "The ancient Romans built aqueducts to carry water across long distances.",
    "Python is a popular programming language for data science and machine learning.",
    "The goalkeeper made an incredible save in the final minutes of the match.",
    "Temperatures in the Arctic have risen faster than the global average.",
    "The restaurant on the corner serves excellent ramen and dumplings.",
]

In [8]:
tokenized_batch = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
).to(MODEL_DEVICE)

with torch.inference_mode():
    _ = model(**tokenized_batch)

# hook now saved the Pythia layer-6 MLP output
layer_6_mlp_activations = captured_activations["layers.6.mlp"]

hook_handle.remove()

# Remove padding positions
real_token_mask = tokenized_batch["attention_mask"].bool()

# Shape: [num_real_tokens, 768]
real_token_activations_cpu = layer_6_mlp_activations[real_token_mask]

print("captured activations:", layer_6_mlp_activations.shape)
print("real token activations on CPU:", real_token_activations_cpu.shape)
print("current device:", real_token_activations_cpu.device)

captured activations: torch.Size([16, 16, 768])
real token activations on CPU: torch.Size([208, 768])
current device: cpu


In [9]:
real_token_activations = real_token_activations_cpu.to(
    DEVICE,
    dtype=sae1.W_dec.dtype,
)

print("activations moved to SAE device:", real_token_activations.device)

activations moved to SAE device: cuda:0


### 4. Encoder Pythia activations with the two SAEs

In [10]:
with torch.inference_mode():
    encoded_A = sae1.encode(real_token_activations)
    encoded_B = sae2.encode(real_token_activations)

# For each token:
# - top_indices stores which 32 of the 32,768 possible SAE features fired
# - top_acts stores the corresponding 32 firing strengths.

top_indices_A = encoded_A.top_indices.detach().cpu()
top_indices_B = encoded_B.top_indices.detach().cpu()
top_acts_A = encoded_A.top_acts.detach().cpu()
top_acts_B = encoded_B.top_acts.detach().cpu()

print("SAE 1 top_indices:", top_indices_A.shape)
print("SAE 2 top_indices:", top_indices_B.shape)
print("SAE 1 top_acts:", top_acts_A.shape)
print("SAE 2 top_acts:", top_acts_B.shape)

SAE 1 top_indices: torch.Size([208, 32])
SAE 2 top_indices: torch.Size([208, 32])
SAE 1 top_acts: torch.Size([208, 32])
SAE 2 top_acts: torch.Size([208, 32])


In [11]:
print("activations:", real_token_activations.shape,
      real_token_activations.device,
      real_token_activations.dtype)

print("SAE 1:", sae1.W_dec.device, sae1.W_dec.dtype)
print("SAE 2:", sae2.W_dec.device, sae2.W_dec.dtype)

activations: torch.Size([208, 768]) cuda:0 torch.float32
SAE 1: cuda:0 torch.float32
SAE 2: cuda:0 torch.float32


### 5. Convert top-k indices into a binary firing matrix

In [12]:
def make_binary_firing_matrix(
    top_indices: torch.Tensor,
    num_feature_columns: int,
) -> torch.Tensor:
    """
    top_indices:
        [num_tokens, 32]

    num_feature_columns:
        Number of possible feature IDs included as columns
        - Full SAE: 32,768 columns
        - This small dev run: only the first D_SUB columns

    Returns firing_matrix:
        [num_tokens, num_feature_columns]
        firing_matrix[t, i] = 1 if feature i fired on token t,
                              0 otherwise.

                              
    32 is the number of active features per token.
    num_feature_columns is the number of possible features represented as columns.
    """
    top_indices = top_indices.cpu().long()
    num_tokens, top_k = top_indices.shape

    firing_matrix = torch.zeros(
        num_tokens,
        num_feature_columns,
        dtype=torch.float32,
    )

    # for this dev subset, ignore selected feature IDs outside D_SUB
    included = top_indices < num_feature_columns

    token_rows = torch.arange(num_tokens).unsqueeze(1).expand(num_tokens, top_k)
    firing_matrix[token_rows[included], top_indices[included]] = 1.0

    return firing_matrix

In [13]:
D_SUB = 2048

A = make_binary_firing_matrix(top_indices_A, D_SUB)  # SAE 1 firing matrix
B = make_binary_firing_matrix(top_indices_B, D_SUB)  # SAE 2 firing matrix

print("A (SAE 1 binary firing matrix):", A.shape)
print("B (SAE 2 binary firing matrix):", B.shape)
print("average included fires per token, SAE 1:", A.sum(dim=1).mean().item())
print("average included fires per token, SAE 2:", B.sum(dim=1).mean().item())

A (SAE 1 binary firing matrix): torch.Size([208, 2048])
B (SAE 2 binary firing matrix): torch.Size([208, 2048])
average included fires per token, SAE 1: 6.899038314819336
average included fires per token, SAE 2: 5.908653736114502


### 6. Compute Jaccard co-occurrence

For feature $i$ in SAE 1 and feature $j$ in SAE 2:


$Jaccard(i, j)$ =
    number of token positions where both features fired
    /
    number of token positions where either feature fired


In [14]:
def compute_jaccard_from_firing_matrices(
    A: torch.Tensor,
    B: torch.Tensor,
):
    """
    A: SAE 1 binary firing matrix, [num_tokens, num_features_A]
    B: SAE 2 binary firing matrix, [num_tokens, num_features_B]

    Returns:
    jaccard_matrix: [num_features_A, num_features_B]
        Fraction of the two features' combined firing set that is shared.

    raw_cooccurrence_matrix: [num_features_A, num_features_B]
        counts tokens where SAE 1 feature i and SAE 2 feature j both fired.

    union_counts: [num_features_A, num_features_B]
        counts tokens where either SAE 1 feature i or SAE 2 feature j fired.

    fire_counts_A: [num_features_A]
        idividual firing count for SAE 1 features.

    fire_counts_B: [num_features_B]
        individual firing count for SAE 2 features.
    """

    raw_cooccurrence_matrix = A.T @ B

    fire_counts_A = A.sum(dim=0)
    fire_counts_B = B.sum(dim=0)

    union_counts = fire_counts_A[:, None] + fire_counts_B[None, :] - raw_cooccurrence_matrix

    jaccard_matrix = torch.nan_to_num(raw_cooccurrence_matrix / union_counts, nan=0.0) # avoid NaNs for features that never fire

    return jaccard_matrix, raw_cooccurrence_matrix, union_counts, fire_counts_A, fire_counts_B

In [15]:
J, C, U, fA, fB = compute_jaccard_from_firing_matrices(A, B)

In [16]:
print("Jaccard matrix:", J.shape)
print("raw co-occurrence matrix:", C.shape)
print("union counts:", U.shape)
print("SAE 1 feature firing counts:", fA.shape)
print("SAE 2 feature firing counts:", fB.shape)

Jaccard matrix: torch.Size([2048, 2048])
raw co-occurrence matrix: torch.Size([2048, 2048])
union counts: torch.Size([2048, 2048])
SAE 1 feature firing counts: torch.Size([2048])
SAE 2 feature firing counts: torch.Size([2048])


### 7. Do Hungarian matching on Jaccard matrix

In [17]:
row_indices, col_indices = linear_sum_assignment(J.numpy(), maximize=True)
matched_scores = J[row_indices, col_indices].numpy()

In [18]:
print("mean matched Jaccard:", matched_scores.mean())
print("median matched Jaccard:", np.median(matched_scores))
print("maximum matched Jaccard:", matched_scores.max())

mean matched Jaccard: 0.047018953
median matched Jaccard: 0.0
maximum matched Jaccard: 1.0


### 8. sanity check

In [19]:
J_self, _, _, fire_counts_self, _ = compute_jaccard_from_firing_matrices(A, A)

active_features = fire_counts_self > 0
active_diagonal = J_self.diag()[active_features]

print("mean self-Jaccard for active features:", active_diagonal.mean().item())
print("minimum self-Jaccard for active features:", active_diagonal.min().item())

mean self-Jaccard for active features: 1.0
minimum self-Jaccard for active features: 1.0
